# Shallow water on the rotating sphere: Galewsky jet

`SphericalShallowWater` solves the shallow water equations on the rotating sphere with the spectral transform method of torch-harmonics, in vorticity–divergence form:

$$
\partial_t \Phi = -\nabla\cdot(\Phi\,\mathbf{u}), \qquad
\partial_t \zeta = -\nabla\cdot\big((\zeta+f)\,\mathbf{u}\big), \qquad
\partial_t \delta = \hat{\mathbf{k}}\cdot\nabla\times\big((\zeta+f)\,\mathbf{u}\big) - \nabla^2\big(\Phi + \tfrac12|\mathbf{u}|^2\big),
$$

with $\Phi = g h$ the geopotential, $\zeta$ the relative vorticity, $\delta$ the divergence, $\mathbf{u}$ the wind and $f$ the Coriolis parameter. Time stepping is third-order Adams–Bashforth with hyperdiffusion as an integrating factor, at L = 256 on a 384 × 768 Gauss–Legendre grid.

The initial condition is the barotropically unstable mid-latitude jet of Galewsky, Scott and Polvani (2004), with a localised height bump and a random height noise. The jet breaks into vortices after a few days. Each simulation samples four parameters:

| Parameter | Range | Paper |
|---|---|---|
| `jet_speed` | 60 to 100 m/s | 80 m/s |
| `height_perturbation_m` | 60 to 180 m | 120 m |
| `perturbation_lon` | 0 to 2π | 0 |
| `perturbation_lat` | π/5 to π/3 | π/4 |

The saved channels are the height perturbation `h` [m], the eastward wind `u_phi` and the southward wind `u_theta` [m/s].

## The simulator

Two simulations run in parallel on the GPU (`batch_size=2`). The run covers 15 days (360 h), saved every 6 h from t = 0 (`skip_nt=0`), so the jet's breakdown is visible; the dataset config drops the first 183 h. The height noise has a root-mean-square of 10 m, as in the `experimental/shallow_water_sphere` config.

The simulator needs a CUDA GPU and the `spherical` extra of autosim, which installs torch-harmonics: `pip install "autosim[spherical]"`.

In [ ]:
import torch
from autosim.experimental.simulations import SphericalShallowWater

sim = SphericalShallowWater(
    lmax=256,
    nlat=384,
    nlon=768,
    T=360.0,              # hours
    dt_save=6.0,          # hours
    skip_nt=0,
    noise_amplitude=10.0, # m
    batch_size=2,
    log_level="warning",
)
print("parameters:", sim.param_names)
print("channels:", sim.output_names)
print("grid:", sim.grid.shape, "snapshots:", sim.n_frames)

## Two simulations with different initial conditions

`forward_samples_spatiotemporal` samples the four parameters uniformly in their ranges, builds one Galewsky initial condition per sample and integrates them. It returns the trajectories in `data`, of shape (simulations, snapshots, 384, 768, 3), and the sampled parameters in `constant_scalars`.

In [ ]:
import time

start = time.time()
batch = sim.forward_samples_spatiotemporal(n=2, random_seed=0)
print(f"{(time.time() - start) / 60:.1f} min")

data = batch["data"]
parameters = batch["constant_scalars"]
print("data:", tuple(data.shape))
for i in range(data.shape[0]):
    values = {name: round(float(parameters[i, k]), 3) for k, name in enumerate(sim.param_names)}
    print(f"simulation {i}:", values)

## Vorticity on the globe

The relative vorticity $\zeta = \hat{\mathbf{k}}\cdot\nabla\times\mathbf{u}$ shows the jet, its breakdown and the vortex filaments. It is not a saved channel, so it is recomputed from the wind with the simulator's vector spherical harmonic transform: `vorticity_divergence` gives the curl coefficients and `to_grid` brings them back to the grid. The wind is passed with component 0 eastward and 1 northward, so `u_theta` (southward) changes sign.

Rows are the two simulations, columns the times 0, 72, 120, 186 and 360 h, on one colour scale.

In [ ]:
import os
import matplotlib.pyplot as plt
from torch_harmonics.plotting import plot_sphere

hours = [0, 72, 120, 186, 360]
snapshots = [round(hour / sim.dt_save) for hour in hours]
n_sims = data.shape[0]

# Wind of the chosen snapshots, (simulations * snapshots, 2, 384, 768): component 0 eastward, 1 northward
u_phi = data[:, snapshots, ..., 1]
u_theta = data[:, snapshots, ..., 2]
wind = torch.stack([u_phi, -u_theta], dim=2).reshape(-1, 2, *sim.grid.shape)
wind = wind.to(device=sim.device, dtype=torch.float64)

vorticity_coefficients, _ = sim.dynamics.vorticity_divergence(wind)
vorticity = sim.dynamics.to_grid(vorticity_coefficients)
vorticity = vorticity.reshape(n_sims, len(snapshots), *sim.grid.shape).cpu()

os.makedirs("figures", exist_ok=True)
limit = vorticity.abs().max().item()
fig = plt.figure(figsize=(20, 4 * n_sims))
panels = fig.subfigures(n_sims, len(snapshots))
for i in range(n_sims):
    for k, hour in enumerate(hours):
        plot_sphere(vorticity[i, k], fig=panels[i, k], projection="orthographic", grid=sim.grid,
                    central_latitude=40, central_longitude=float(parameters[i, 2]) * 180 / torch.pi,
                    cmap="twilight_shifted", vmin=-limit, vmax=limit,
                    title=f"simulation {i}, t = {hour} h")
fig.savefig("figures/vorticity.png", dpi=60, bbox_inches="tight")
plt.close(fig)
print("saved figures/vorticity.png")

In [ ]:
from IPython.display import Image, display

display(Image("figures/vorticity.png"))

## The saved channels

The three saved channels of both simulations at the last snapshot (t = 360 h), on a Robinson map of the whole sphere. The height and the winds are smoother than the vorticity, which is a derivative of the wind.

In [ ]:
last = data[:, -1]  # (simulations, 384, 768, 3)
channel_titles = ["h [m]", "u_phi [m/s]", "u_theta [m/s]"]

fig = plt.figure(figsize=(18, 4 * n_sims))
panels = fig.subfigures(n_sims, len(channel_titles))
for i in range(n_sims):
    for c, title in enumerate(channel_titles):
        field = last[:, ..., c]
        limit = field.abs().max().item()
        vmin = field.min().item() if c == 0 else -limit
        vmax = field.max().item() if c == 0 else limit
        plot_sphere(field[i], fig=panels[i, c], projection="robinson", grid=sim.grid,
                    cmap="RdBu_r", vmin=vmin, vmax=vmax, title=f"simulation {i}, {title}, t = 360 h")
fig.savefig("figures/channels.png", dpi=60, bbox_inches="tight")
plt.close(fig)
print("saved figures/channels.png")

In [ ]:
display(Image("figures/channels.png"))